In [1]:
import grain
import jax
from context_flux_no.data import TheWellDataSource


jax.config.update("jax_default_device", jax.devices("gpu")[3])

In [2]:
jax.devices()

[CudaDevice(id=0),
 CudaDevice(id=1),
 CudaDevice(id=2),
 CudaDevice(id=3),
 CudaDevice(id=4),
 CudaDevice(id=5),
 CudaDevice(id=6)]

In [6]:
source = TheWellDataSource(
    "../../data/datasets",
    "euler_multi_quadrants_periodicBC",
    include_filters=["gamma_1.3_CO2"],
    window_size=11,
    downsample_spatial=8,
    exclude_field_names=["pressure"],
)
sampler = grain.samplers.IndexSampler(len(source), shuffle=True, seed=0)
transforms = [grain.transforms.Batch(batch_size=64)]

In [8]:
dataloader = grain.DataLoader(
    data_source=source,
    sampler=sampler,
    operations=transforms,
    worker_count=6,
    # read_options=grain.ReadOptions(num_threads=32, prefetch_buffer_size=1000),
    shard_options=grain.sharding.ShardByJaxProcess(),
)

In [9]:
dataiter = iter(dataloader)
batch = next(dataiter)

In [10]:
type(batch)

numpy.ndarray

In [14]:
batch

array([[[[[ 1.00840366e+00,  9.66938436e-01,  9.06345963e-01, ...,
            1.12304592e+00,  1.08652139e+00,  1.04929972e+00],
          [ 9.85003471e-01,  9.30810034e-01,  8.56592596e-01, ...,
            1.11725318e+00,  1.07335579e+00,  1.02959549e+00],
          [ 9.55441535e-01,  8.79037261e-01,  7.80157864e-01, ...,
            1.11364746e+00,  1.06240177e+00,  1.01052296e+00],
          ...,
          [ 1.08218670e+00,  1.08871317e+00,  1.41180193e+00, ...,
            5.93911707e-01,  1.07323849e+00,  1.10266221e+00],
          [ 1.04312968e+00,  1.00716305e+00,  1.00613761e+00, ...,
            1.14267647e+00,  1.11200631e+00,  1.07765841e+00],
          [ 1.02499473e+00,  9.86859560e-01,  9.43559825e-01, ...,
            1.12733006e+00,  1.09795868e+00,  1.06274283e+00]],

         [[ 9.30732787e-01,  8.87470484e-01,  8.21874082e-01, ...,
            1.07935882e+00,  1.02668238e+00,  9.78281498e-01],
          [ 9.13118005e-01,  8.52871478e-01,  7.69949973e-01, ...,
      

In [ ]:
ds = (
    grain.MapDataset.source(source)
    .seed(seed=0)
    .shuffle()
    .batch(batch_size=64)
)
# it = iter(ds)

# for _ in range(1):
#     next(it)

In [6]:
performance_config = grain.experimental.pick_performance_config(
    ds=ds, ram_budget_mb=10000, max_workers=8, max_buffer_size=None
)

In [7]:
performance_config

PerformanceConfig(multiprocessing_options=MultiprocessingOptions(num_workers=8, per_worker_buffer_size=1, enable_profiling=False), read_options=ReadOptions(num_threads=16, prefetch_buffer_size=227))